<a href="https://colab.research.google.com/github/organichotdog/micrograd/blob/main/micrograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Value:

  def __init__(self, value, children = (), operation = '', grad = 0):
    self.data = value
    self.children = children
    self.operation = operation
    self._backward = lambda: None
    self.grad = grad


  def __repr__(self):
    return f"{self.data}, {self.children}, {self.operation}"


  def __add__(self, val):
    val = Value(val) if not isinstance(val, Value) else val
    output = Value(self.data + val.data, (self, val), '+')

    def _backward():
      self.grad += output.grad
      val.grad += output.grad

    output._backward = _backward
    return output


  def __mul__(self, val):
    val = Value(val) if not isinstance(val, Value) else val
    output = Value(self.data * val.data, (self, val), '*')

    def _backward():
      self.grad += val.data * output.grad
      val.grad += self.data * output.grad

    output._backward = _backward
    return output


  def __pow__(self, exp):
    output = Value(self.data ** exp, (self,), '**')

    def _backward():
      self.grad += exp * (self.data ** (exp - 1)) * output.grad

    output._backward = _backward
    return output


  def __radd__(self, val):
    return self + val


  def __rmul__(self, val):
    return self * val


  def __neg__(self):
    return self * -1


  def __sub__(self, val):
    return self + (-val)


  def __rsub__(self, val):
    return val + (-self)


  def __truediv__(self, val):
    return self * (val ** -1)


  def __rtruediv__(self, val):
    return val * (self ** -1)


  def backward(self):
    graph = []
    def ordering(self):
      if self not in graph:
        graph.append(self)
        for child in self.children:
          ordering(child)

    ordering(self)

    self.grad = 1
    for node in graph:
      node._backward()


In [ ]:
a = Value(2)
b = Value(3)

c = a * b
d = c + a
e = c + b

f = d * e

f.backward()

print(a.grad)
print(b.grad)

12
14
